# Cost-Aware No-Trade Band Test (k=10)

Sequential two-pointer swap rule: swap incumbent for candidate only if
`Û(candidate) ≥ Û(incumbent) + 10 bps` (long) or `≤ Û(incumbent) - 10 bps` (short).

**Compared score families:**
1. U-only (rank by `Û`)
2. Z-score composite (`0.5·z(P) + 0.5·z(U)` for selection; **raw Û** for swap ranking)
3. P-gate 0.03 (P-filter + rank by Û)
4. P-gate 0.05 (P-filter + rank by Û)

**Models:** DNN, XGB, RF, ENS1  |  **k = 10, cost = 5 bps/half-turn, swap threshold = 10 bps**

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..') / 'src'))
from krauss.backtest.no_trade_band import backtest_with_band

ROOT = Path('..')
PROCESSED = ROOT / 'data' / 'processed'

pred = pd.read_parquet(PROCESSED / 'predictions_phase2.parquet')
returns = pd.read_parquet(PROCESSED / 'daily_returns.parquet')
pred['date'] = pd.to_datetime(pred['date'])
returns['date'] = pd.to_datetime(returns['date'])

MODELS = ['dnn', 'xgb', 'rf', 'ens1']
K = 10

print(f'{len(pred):,} prediction rows, {pred["date"].nunique():,} dates')

2,852,210 prediction rows, 5,750 dates


In [2]:
# Precompute derived score columns per model
def xsec_z(series, date):
    grp = series.groupby(date)
    return (series - grp.transform('mean')) / grp.transform('std')

for m in MODELS:
    p_col = f'p_{m}'
    u_col = f'u_{m}'

    # Z-score composite = 0.5*z(P) + 0.5*z(U)
    zp = xsec_z(pred[p_col], pred['date'])
    zu = xsec_z(pred[u_col], pred['date'])
    pred[f'zcomp_{m}'] = 0.5 * zp + 0.5 * zu

    # P-gate masks: score = Û if P passes the gate, else NaN
    for thresh_str, thresh in [('003', 0.03), ('005', 0.05)]:
        pred[f'gate_long_{thresh_str}_{m}'] = pred[u_col].where(pred[p_col] > 0.5 + thresh)
        pred[f'gate_short_{thresh_str}_{m}'] = pred[u_col].where(pred[p_col] < 0.5 - thresh)

print('Score columns precomputed.')

Score columns precomputed.


In [3]:
# Build (long_score_col, short_score_col) per strategy
def strategy_cols(model, family):
    u = f'u_{model}'
    if family == 'U-only':
        return f'score_u_{model}', f'score_u_{model}', u
    if family == 'Z-comp':
        return f'zcomp_{model}', f'zcomp_{model}', u
    if family == 'Gate 0.03':
        return f'gate_long_003_{model}', f'gate_short_003_{model}', u
    if family == 'Gate 0.05':
        return f'gate_long_005_{model}', f'gate_short_005_{model}', u
    raise ValueError(family)

FAMILIES = ['U-only', 'Z-comp', 'Gate 0.03', 'Gate 0.05']

# Baseline = 'always swap' (very negative threshold). With-band = 10 bps.
BAND_BPS = {'Baseline': -1e9, 'Band 10bps': 10.0}

def metrics(daily):
    r = daily['port_ret_net']
    mu, sd = r.mean(), r.std()
    return {
        'Daily net (%)':    mu * 100,
        'Ann. return (%)':  mu * 252 * 100,
        'Sharpe (net)':     (mu / sd) * np.sqrt(252) if sd > 0 else np.nan,
        'Turnover':         daily['turnover'].mean(),
        'n_long avg':       daily['n_long'].mean(),
        'n_short avg':      daily['n_short'].mean(),
    }

rows = []
for model in MODELS:
    for fam in FAMILIES:
        long_col, short_col, u_col = strategy_cols(model, fam)
        for band_name, thresh in BAND_BPS.items():
            out = backtest_with_band(
                predictions=pred, returns=returns, k=K,
                long_score_col=long_col, short_score_col=short_col,
                u_col=u_col,
                half_turn_bps=5.0, swap_threshold_bps=thresh,
            )
            row = {'Model': model.upper(), 'Family': fam, 'Band': band_name}
            row.update(metrics(out['daily']))
            rows.append(row)
            print(f'{model:5s} {fam:10s} {band_name:12s} '
                  f'net={row["Daily net (%)"]:+.4f}%  '
                  f'Sh={row["Sharpe (net)"]:+.2f}  '
                  f'turn={row["Turnover"]:.3f}')

res = pd.DataFrame(rows)
print('\nDone.')

dnn   U-only     Baseline     net=+0.0822%  Sh=+0.52  turn=1.205
dnn   U-only     Band 10bps   net=-0.0037%  Sh=-0.03  turn=0.021
dnn   Z-comp     Baseline     net=+0.1218%  Sh=+0.79  turn=1.198
dnn   Z-comp     Band 10bps   net=+0.0039%  Sh=+0.03  turn=0.021
dnn   Gate 0.03  Baseline     net=+0.0298%  Sh=+0.13  turn=0.449
dnn   Gate 0.03  Band 10bps   net=-0.0073%  Sh=-0.05  turn=0.009
dnn   Gate 0.05  Baseline     net=+0.0603%  Sh=+0.23  turn=0.215
dnn   Gate 0.05  Band 10bps   net=+0.0059%  Sh=+0.03  turn=0.005
xgb   U-only     Baseline     net=+0.1829%  Sh=+1.47  turn=2.696
xgb   U-only     Band 10bps   net=+0.2078%  Sh=+1.68  turn=1.859
xgb   Z-comp     Baseline     net=+0.2242%  Sh=+1.80  turn=2.686
xgb   Z-comp     Band 10bps   net=+0.2428%  Sh=+1.92  turn=1.659
xgb   Gate 0.03  Baseline     net=+0.2073%  Sh=+1.65  turn=2.463
xgb   Gate 0.03  Band 10bps   net=+0.2256%  Sh=+1.74  turn=1.493
xgb   Gate 0.05  Baseline     net=+0.2542%  Sh=+1.37  turn=2.034
xgb   Gate 0.05  Band 10b

In [4]:
# Side-by-side: Baseline vs Band, per model × family
def side_by_side(metric):
    wide = res.pivot_table(index=['Model', 'Family'], columns='Band', values=metric)
    wide = wide[['Baseline', 'Band 10bps']]
    wide['Δ'] = wide['Band 10bps'] - wide['Baseline']
    return wide

for metric in ['Daily net (%)', 'Ann. return (%)', 'Sharpe (net)']:
    tbl = side_by_side(metric)
    fmt = '{:+.4f}' if 'Daily' in metric else ('{:+.2f}' if 'Sharpe' in metric else '{:+.2f}')
    display(tbl.style.format(fmt)
            .background_gradient(cmap='RdYlGn', subset=['Δ'])
            .set_caption(f'{metric} — Baseline vs Band (post-cost, k=10)'))

In [5]:
# Turnover reduction
tur = res.pivot_table(index=['Model', 'Family'], columns='Band', values='Turnover')
tur = tur[['Baseline', 'Band 10bps']]
tur['% reduction'] = (1 - tur['Band 10bps'] / tur['Baseline']) * 100
display(tur.style.format({'Baseline': '{:.3f}', 'Band 10bps': '{:.3f}', '% reduction': '{:.1f}%'})
        .background_gradient(cmap='Greens', subset=['% reduction'])
        .set_caption('Daily turnover — Baseline vs Band (k=10)'))

In [6]:
# Flat ranking: best Sharpe overall, across all 32 (model × family × band)
top = res.sort_values('Sharpe (net)', ascending=False).head(12)
cols = ['Model', 'Family', 'Band', 'Daily net (%)', 'Ann. return (%)', 'Sharpe (net)', 'Turnover']
display(top[cols].style.format({
    'Daily net (%)': '{:+.4f}',
    'Ann. return (%)': '{:+.2f}',
    'Sharpe (net)': '{:+.2f}',
    'Turnover': '{:.3f}',
}).background_gradient(cmap='RdYlGn', subset=['Sharpe (net)'])
  .set_caption('Top 12 configurations by post-cost Sharpe'))

,Model,Family,Band,Daily net (%),Ann. return (%),Sharpe (net),Turnover
26,ENS1,Z-comp,Baseline,+0.3058,+77.06,+1.97,2.390
11,XGB,Z-comp,Band 10bps,+0.2428,+61.20,+1.92,1.659
18,RF,Z-comp,Baseline,+0.2862,+72.12,+1.92,2.365
27,ENS1,Z-comp,Band 10bps,+0.2900,+73.08,+1.85,1.032
10,XGB,Z-comp,Baseline,+0.2242,+56.51,+1.80,2.686
13,XGB,Gate 0.03,Band 10bps,+0.2256,+56.86,+1.74,1.493
15,XGB,Gate 0.05,Band 10bps,+0.2552,+64.30,+1.72,1.302
24,ENS1,U-only,Baseline,+0.2567,+64.68,+1.72,2.288
25,ENS1,U-only,Band 10bps,+0.2561,+64.54,+1.71,1.132
20,RF,Gate 0.03,Baseline,+0.2633,+66.35,+1.68,2.390
